In [ ]:
import os

import torch
import trimesh
import numpy as np
from torch.utils.data import Dataset
from PIL import Image

class MassHuman(Dataset):
    def __init__(self, root_dir="mass_humans"):
        self.root_dir = root_dir

    def __len__(self):
        return len([file for file in os.listdir(self.root_dir) if file.endswith(".obj")])

    def _load_obj(self, obj_file_path):
        resolver = trimesh.resolvers.FilePathResolver(os.path.dirname(obj_file_path))
        loaded_data = trimesh.load(obj_file_path, force=None, resolver=resolver)
        meshes_to_process = []
        for _, geom in loaded_data.geometry.items():
            if isinstance(geom, trimesh.Trimesh):
                meshes_to_process.append(geom)

        sorted_meshs = sorted(meshes_to_process, key=lambda m: len(m.faces), reverse=True)
        return sorted_meshs[0]

    def _split_human_mesh(self, mesh, test_texture):
        main_body_face_indices = []
        non_main_body_face_indices = []

        for i, face in enumerate(mesh.faces):
            avg_uv = np.mean(mesh.visual.uv[face], axis=0)
            pixel_x = int(avg_uv[0] * test_texture.width) % test_texture.width
            pixel_y = int((1 - avg_uv[1]) * test_texture.height) % test_texture.height

            color = test_texture.getpixel((pixel_x, pixel_y))
            if np.all(np.array(color) == 0):
                main_body_face_indices.append(i)
            elif np.all(np.array(color) == 255):
                non_main_body_face_indices.append(i)
            else:
                raise ValueError(f"Unexpected color {color} at UV coordinates {avg_uv}")

        return non_main_body_face_indices

    def __getitem__(self, idx):
        test_texture = Image.open("test_texture.png")
        human_mesh = self._load_obj(f"{self.root_dir}/human_{idx}.obj")
        human_mesh.merge_vertices()
        non_main_body_face_indices = self._split_human_mesh(human_mesh, test_texture)
        return torch.tensor(non_main_body_face_indices, dtype=torch.long)


dataset = MassHuman()
dataloader = torch.utils.data.DataLoader(dataset, num_workers=32, shuffle=True, in_order=False)

In [ ]:
from tqdm import tqdm

# Check all samples
his = None
for batch in tqdm(dataloader):
    this = batch[0]
    if his is not None:
        if not torch.equal(this, his):
            raise ValueError("Inconsistent non-main-body face indices across samples.")
    his = this

np.save("non_main_body_face_indices.npy", his.numpy())